# Phase 5 — GRPO seeds 42 + 1234 (the tracer recipe, frozen)

**Runtime: A100.** ~1.5 h per seed (~20 units each). **Checkpointed per seed** —
you can run seed 42, stop, and rerun later for 1234; finished seeds are skipped.

Identical to the notebook-08 tracer in every way except the seed (Amendment A3
paired lineage: each seed inits from ITS OWN merged SFT checkpoint). The tracer
was healthy (no hacking, +2.3 pass@1, pass@16 up to 95.1), so nothing is tuned —
recipe frozen for comparability across lineages.

Per seed: quick 20-bug advisory gate → 250 GRPO steps (G=8, temp 1.0, lr 5e-6,
β 0.04, hardened sentinel runner, canonical src/reward.py) → dev eval →
restraint probe. Final cell prints the full 3-arm × 3-seed dev table.

In [ ]:
# Setup: Drive, repo, canonical modules
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc, uuid
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
PHASE4 = '/content/drive/MyDrive/rl-post-training/phase4'
PHASE5 = '/content/drive/MyDrive/rl-post-training/phase5'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts', 'reward', 'variance_gate'):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
import reward as rw
import variance_gate as vg
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
train_bugs = [b for b in bugs if b['split'] == 'train']
rl_bugs = [b for b in train_bugs if routing.get(b['id']) == 'rl']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
dev_clean = [r for r in restraint if r['split'] == 'dev']
print(len(rl_bugs), 'RL-pile bugs |', len(dev_bugs), 'dev |', len(dev_clean), 'dev clean')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# Hardened runner (sentinel), canonical reward, hack log (per-seed path)
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
SENTINEL = uuid.uuid4().hex
HACK_LOG = ['']   # set per seed in run_seed
_call_n = [0]

def run_tests(code, rec, timeout=10):
    tests = list(rec['test_list'])
    try:
        compile(code, '<cand>', 'exec'); compiles = True
    except Exception:
        return (False, 0, len(tests))
    harness = (
        '\n'.join(list(rec['test_imports'])) + '\n' + code + '\n'
        + '__zz_pass = 0\n'
        + f'for __zz_t in {tests!r}:\n'
        + '    try:\n        exec(__zz_t)\n        __zz_pass += 1\n'
        + '    except BaseException:\n        pass\n'
        + f'print("{SENTINEL}", __zz_pass)\n'
    )
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(harness); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, timeout=timeout)
        n_passed = 0
        for line in r.stdout.decode(errors='ignore').splitlines():
            parts = line.split()
            if len(parts) == 2 and parts[0] == SENTINEL and parts[1].isdigit():
                n_passed = int(parts[1])
    except subprocess.TimeoutExpired:
        n_passed = 0
    finally:
        os.unlink(path)
    return (compiles, n_passed, len(tests))

def grpo_reward(prompts, completions, test_imports, test_list, **kw):
    recs = [{'test_imports': ti, 'test_list': tl} for ti, tl in zip(test_imports, test_list)]
    codes = [extract_code(c) for c in completions]
    with ThreadPoolExecutor(max_workers=8) as pool:
        outs = list(pool.map(lambda a: run_tests(*a), zip(codes, recs)))
    rewards = [rw.reward(rw.ExecResult(compiles=c, num_passed=p, num_tests=t,
                                       output_tokens=len(comp) // 4))
               for (c, p, t), comp in zip(outs, completions)]
    _call_n[0] += 1
    best = max(range(len(rewards)), key=lambda i: rewards[i])
    worst = min(range(len(rewards)), key=lambda i: rewards[i])
    with open(HACK_LOG[0], 'a') as f:
        f.write(json.dumps({'call': _call_n[0], 'mean_r': sum(rewards)/len(rewards),
                            'n_pass': sum(1 for r in rewards if r >= 1.25),
                            'best_snippet': codes[best][:200],
                            'worst_snippet': codes[worst][:200]}) + '\n')
    return rewards

# self-test (same as the tracer): hack MUST show passed=0
_demo = {'test_imports': [], 'test_list': ['assert add2(2) == 4', 'assert add2(0) == 2']}
for name, cand in (('good', 'def add2(x):\n    return x + 2'),
                   ('HACK sys.exit(0)', 'import sys\nsys.exit(0)\ndef add2(x):\n    return x')):
    c, p, t = run_tests(cand, _demo)
    print(f"{name:<18} compiles={c} passed={p}/{t}")

In [ ]:
# Eval helpers (identical protocol to 05/07/08)
import torch
from unsloth import FastLanguageModel
try:
    from unsloth import PatchFastRL
    PatchFastRL('GRPO', FastLanguageModel)
except ImportError:
    pass
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

def chat(tok, p):
    return tok.apply_chat_template([{'role': 'user', 'content': p}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def sample_k(model, tok, source_code, test_list, k):
    enc = tok([chat(tok, build_repair_prompt(source_code, test_list))],
              return_tensors='pt', padding=True, padding_side='left').to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=k, max_new_tokens=512,
                         pad_token_id=tok.eos_token_id)
    return [extract_code(t) for t in tok.batch_decode(
        out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

def passes_all(code, rec):
    c, p, t = run_tests(code, rec)
    return p == t

def dev_eval(model, tok, tag, k=16):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_bugs:
        codes = sample_k(model, tok, b['buggy'], b['test_list'], k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes_all(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, 'pass@16': p16, 'gap': p16 - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE5}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@16={p16*100:.1f}%  gap={(p16-p1)*100:.1f}")
    return res

def restraint_probe(model, tok, tag, k=4):
    FastLanguageModel.for_inference(model)
    norm = lambda s: ' '.join(s.split())
    sp = un = tot = 0
    for r in dev_clean:
        codes = sample_k(model, tok, r['code'], r['test_list'], k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes_all(c, r), codes))
        sp += sum(flags); un += sum(norm(c) == norm(r['code']) for c in codes); tot += k
    json.dump({'tag': tag, 'still_pass': sp/tot, 'unchanged': un/tot},
              open(f'{PHASE5}/restraint_probe_{tag}.json', 'w'))
    print(f"[probe {tag}]  still passes: {sp/tot*100:.1f}%  unchanged: {un/tot*100:.1f}%")

In [ ]:
# GRPO per seed — frozen tracer recipe; skip-if-done; advisory 20-bug gate
def run_seed(seed):
    print(f'===== GRPO SEED {seed} (init: sft_notrace_s{seed}_ep1) =====')
    HACK_LOG[0] = f'{PHASE5}/hacking_watch_s{seed}.jsonl'
    merged = f'/content/merged_s{seed}'
    if not os.path.exists(merged):
        m, t = FastLanguageModel.from_pretrained(
            f'{PHASE3}/sft_notrace_s{seed}_ep1', max_seq_length=1536,
            load_in_4bit=False, dtype=torch.bfloat16)
        m.save_pretrained_merged(merged, t, save_method='merged_16bit')
        del m, t; gc.collect(); torch.cuda.empty_cache()
    model, tok = FastLanguageModel.from_pretrained(
        merged, max_seq_length=1536, load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=32, lora_alpha=64, lora_dropout=0,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=seed)
    # advisory gate: 20 bugs x 8
    FastLanguageModel.for_inference(model)
    pc = []
    for b in random.Random(seed).sample(rl_bugs, 20):
        codes = sample_k(model, tok, b['buggy'], b['test_list'], 8)
        with ThreadPoolExecutor(max_workers=8) as pool:
            outs = list(pool.map(lambda c: passes_all(c, b), codes))
        pc.append((b['id'], sum(outs)))
    g = vg.gate(pc, group_size=8)
    print(f"gate (advisory, n=20): learnable {g['learnable_fraction']*100:.0f}% (tracer was 52%)")
    # train
    rows = [{'prompt': chat(tok, build_repair_prompt(b['buggy'], b['test_list'])),
             'test_imports': list(b['test_imports']), 'test_list': list(b['test_list'])}
            for b in rl_bugs]
    ds = Dataset.from_list(rows).shuffle(seed=seed)
    FastLanguageModel.for_training(model)
    cfg = GRPOConfig(
        per_device_train_batch_size=8, gradient_accumulation_steps=4,
        num_generations=8, max_prompt_length=768, max_completion_length=512,
        temperature=1.0, top_p=0.95, beta=0.04,
        learning_rate=5e-6, lr_scheduler_type='cosine', warmup_ratio=0.1,
        max_steps=250, logging_steps=10, seed=seed, output_dir='/content/out',
        report_to='none', save_strategy='no')
    trainer = GRPOTrainer(model=model, processing_class=tok, reward_funcs=grpo_reward,
                          args=cfg, train_dataset=ds)
    trainer.train()
    model.save_pretrained(f'{PHASE5}/grpo_s{seed}')
    dev_eval(model, tok, f'grpo_seed{seed}')
    restraint_probe(model, tok, f'grpo_s{seed}')
    del trainer, model, tok
    gc.collect(); torch.cuda.empty_cache()

for s in (42, 1234):
    if os.path.exists(f'{PHASE5}/dev_eval_grpo_seed{s}.json'):
        print(f'seed {s} already done, skipping'); continue
    run_seed(s)

In [ ]:
# THE FULL DEV TABLE: 3 arms x 3 seeds + means (the study's core dev result)
import statistics as st
SEEDS = (3407, 42, 1234)
arms = {
    'SFT':  [f'{PHASE3}/dev_eval_ep1_seed{s}.json' for s in SEEDS],
    'DPO':  [f'{PHASE4}/dev_eval_dpo_ep1_seed{s}.json' for s in SEEDS],
    'GRPO': [f'{PHASE5}/dev_eval_grpo_seed{s}.json' for s in SEEDS],
}
print(f"{'arm':<6} {'seed':<6} pass@1   pass@16   gap")
summary = {}
for arm, paths in arms.items():
    rows = []
    for s, p in zip(SEEDS, paths):
        if not os.path.exists(p):
            print(f'{arm:<6} {s:<6} (not run yet)'); continue
        r = json.load(open(p)); rows.append(r)
        print(f"{arm:<6} {s:<6} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
    if len(rows) == 3:
        p1 = [r['pass@1']*100 for r in rows]; p16 = [r['pass@16']*100 for r in rows]
        summary[arm] = (st.mean(p1), st.stdev(p1), st.mean(p16))
print()
for arm, (m1, s1, m16) in summary.items():
    print(f'{arm:<6} mean pass@1 {m1:.1f}% (sd {s1:.1f})   mean pass@16 {m16:.1f}%')
print('\nRuler: dev tie threshold ~1.5 pts. Paired per-seed deltas matter more')
print('than means. Bring the whole printout back — next is the random-reward')
print('control, then the Phase-7 final exam.')

## Bring back to the session
1. Both seeds' **gate lines** and a few **reward-table rows** each
2. The **full 3×3 dev table + means**
3. The **restraint probe** lines
4. Anything odd in the hacking watch files

After this: random-reward control (Phase 6), then the one-shot Phase-7 final
exam for all arms, then the write-up.